In [2]:
import os
import duckdb
from dotenv import load_dotenv

load_dotenv()

pg_host = os.getenv('PG_HOST')
pg_port = os.getenv('PG_PORT')
pg_dbname = os.getenv('PG_DBNAME')
pg_user = os.getenv('PG_USER')
pg_password = os.getenv('PG_PASSWORD')

duck_con = duckdb.connect("musicbrainz.duckdb")

duck_con.execute("""
INSTALL postgres;
LOAD postgres;
""")

duck_con.execute(f"""
ATTACH IF NOT EXISTS 'host={pg_host} port={pg_port} dbname={pg_dbname} user={pg_user} password={pg_password}'
AS mb_pg
(TYPE postgres, READ_ONLY, SCHEMA {pg_user});
""")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()

DUCKDB_PATH = ROOT / "musicbrainz.duckdb"
SQL_PATH = ROOT / "sql_features" / "queries" / "mb_album_stats_duckdb.sql"

In [17]:
duck_con.execute("USE main;")

sql = SQL_PATH.read_text(encoding="utf-8-sig")
duck_con.execute(sql)



In [20]:
df = duck_con.sql("""
SELECT
    COUNT(*)                                                AS total_albums,
    SUM((first_release_year IS NULL)::INT)                  AS missing_year,
    SUM((track_count = 0)::INT)                             AS no_tracks,
    SUM((track_count_with_length = 0
         AND track_count > 0)::INT)                         AS tracks_but_no_lengths,
    ROUND(AVG(track_count),             2)                  AS avg_track_count,
    ROUND(AVG(mean_length_ms / 1000.0), 2)                  AS avg_mean_track_length_sec,
    ROUND(AVG(median_length_ms / 1000.0), 2)                AS avg_median_track_length_sec,
    ROUND(AVG(total_length_ms / 60000.0), 2)                AS avg_album_length_min
FROM album_stats;
""").show()

┌──────────────┬──────────────┬───────────┬───────────────────────┬─────────────────┬───────────────────────────┬─────────────────────────────┬──────────────────────┐
│ total_albums │ missing_year │ no_tracks │ tracks_but_no_lengths │ avg_track_count │ avg_mean_track_length_sec │ avg_median_track_length_sec │ avg_album_length_min │
│    int64     │    int128    │  int128   │        int128         │     double      │          double           │           double            │        double        │
├──────────────┼──────────────┼───────────┼───────────────────────┼─────────────────┼───────────────────────────┼─────────────────────────────┼──────────────────────┤
│      2235464 │       120188 │         0 │                213392 │           15.02 │                    314.66 │                      305.74 │                61.96 │
└──────────────┴──────────────┴───────────┴───────────────────────┴─────────────────┴───────────────────────────┴─────────────────────────────┴──────────────────────

In [21]:
PARQUET_PATH = ROOT / "data" / "sql_feature_album_track_stats.parquet"

duck_con.execute(f"""
COPY album_stats
TO '{PARQUET_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

In [22]:
df = duck_con.sql("""
SELECT *
FROM album_stats
""").df()

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2235464 entries, 0 to 2235463
Data columns (total 17 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   release_group_id         int32  
 1   first_release_year       Int32  
 2   medium_count             int16  
 3   track_count              int16  
 4   track_count_with_length  int16  
 5   pct_tracks_with_length   float32
 6   total_length_ms          Int64  
 7   mean_length_ms           Int32  
 8   median_length_ms         Int32  
 9   stddev_length_ms         Int32  
 10  variance_length_ms       Int64  
 11  min_length_ms            Int32  
 12  max_length_ms            Int32  
 13  range_length_ms          Int32  
 14  p25_length_ms            Int32  
 15  p75_length_ms            Int32  
 16  iqr_length_ms            Int32  
dtypes: Int32(10), Int64(2), float32(1), int16(3), int32(1)
memory usage: 174.8 MB


In [25]:
df[(df['first_release_year'].isna()== True)].shape[0]

120188

In [27]:
df = pd.read_parquet('../data/sql_feature_album_track_stats.parquet')

In [28]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
release_group_id,2235464.0,2.178496e+06,1.310953e+06,2.0,1.078664e+06,2.050166e+06,3.246550e+06,4.772411e+06
first_release_year,2115276.0,2.007322e+03,1.337004e+01,1884.0,2.000000e+03,2.010000e+03,2.018000e+03,2.923000e+03
medium_count,2235464.0,1.146284e+00,1.144460e+00,1.0,1.000000e+00,1.000000e+00,1.000000e+00,2.660000e+02
track_count,2235464.0,1.502134e+01,2.066735e+01,1.0,9.000000e+00,1.200000e+01,1.600000e+01,6.666000e+03
track_count_with_length,2235464.0,1.354341e+01,2.036961e+01,0.0,8.000000e+00,1.100000e+01,1.500000e+01,6.666000e+03
pct_tracks_with_length,2235464.0,8.991222e+01,2.980284e+01,0.0,1.000000e+02,1.000000e+02,1.000000e+02,1.000000e+02
total_length_ms,2022072.0,3.717872e+06,1.382363e+07,1.0,2.292172e+06,2.932000e+06,3.942564e+06,1.540765e+10
mean_length_ms,2022072.0,3.146609e+05,1.740630e+06,1.0,1.948500e+05,2.373950e+05,3.004170e+05,2.096661e+09
median_length_ms,2022072.0,3.057423e+05,1.743889e+06,1.0,1.902930e+05,2.321330e+05,2.907792e+05,2.096661e+09
stddev_length_ms,2022072.0,8.107593e+04,2.731428e+05,0.0,3.579400e+04,5.564400e+04,8.911300e+04,2.381016e+08
